In [ ]:
from imageio import imread
from scipy.ndimage import gaussian_filter, label, center_of_mass

# =========================
# USER PARAMETERS
# =========================

# File lists (can be multiple for averaging)
atoms_files = ["atoms1.png", "atoms2.png"]
ref_files   = ["ref1.png", "ref2.png"]
dark_files  = ["dark1.png", "dark2.png"]

# Calibration
atom_per_OD = 46.0

# Processing parameters
epsilon = 1e-6
threshold_OD = 0.05        # threshold to remove noise
blur_sigma = 2             # smoothing for detection
roi_half_size = 80         # pixels around detected center

# =========================
# HELPER FUNCTIONS
# =========================

def load_and_average(file_list):
    imgs = [imread(f).astype(float) for f in file_list]
    
    # Convert RGB → grayscale if needed
    imgs = [img.mean(axis=2) if img.ndim == 3 else img for img in imgs]
    
    return np.mean(imgs, axis=0)

# =========================
# LOAD & AVERAGE IMAGES
# =========================

atoms = load_and_average(atoms_files)
ref   = load_and_average(ref_files)
dark  = load_and_average(dark_files)

# =========================
# COMPUTE OD
# =========================

numerator   = atoms - dark
denominator = ref   - dark

numerator[numerator <= 0] = epsilon
denominator[denominator <= 0] = epsilon

OD = -np.log(numerator / denominator)

# =========================
# AUTOMATIC MOT DETECTION
# =========================

# Smooth to suppress noise
OD_smooth = gaussian_filter(OD, blur_sigma)

# Threshold to find cloud
mask = OD_smooth > threshold_OD

# Label connected regions
labeled, num_features = label(mask)

if num_features == 0:
    raise RuntimeError("No MOT detected. Adjust threshold.")

# Find largest region (assume MOT is biggest)
sizes = [(labeled == i).sum() for i in range(1, num_features+1)]
largest_label = np.argmax(sizes) + 1

mot_mask = (labeled == largest_label)

# Find center of mass
cy, cx = center_of_mass(mot_mask)

cy, cx = int(cy), int(cx)

print(f"Detected MOT center: (y={cy}, x={cx})")

# =========================
# DEFINE ROI AROUND MOT
# =========================

ymin = max(cy - roi_half_size, 0)
ymax = min(cy + roi_half_size, OD.shape[0])
xmin = max(cx - roi_half_size, 0)
xmax = min(cx + roi_half_size, OD.shape[1])

OD_crop = OD[ymin:ymax, xmin:xmax]

# =========================
# THRESHOLDING
# =========================

OD_crop_thresh = OD_crop.copy()
OD_crop_thresh[OD_crop_thresh < threshold_OD] = 0

# =========================
# ATOM NUMBER
# =========================

OD_sum = np.sum(OD_crop_thresh)
N_atoms = atom_per_OD * OD_sum

print(f"Sum of OD (thresholded): {OD_sum:.3e}")
print(f"Estimated atom number: {N_atoms:.3e}")

# =========================
# VISUALIZATION
# =========================

plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
plt.title("OD Image")
plt.imshow(OD)
plt.scatter(cx, cy, color='red', s=50)
plt.colorbar()

plt.subplot(1,3,2)
plt.title("Detected MOT Region")
plt.imshow(mot_mask)
plt.colorbar()

plt.subplot(1,3,3)
plt.title("ROI (Thresholded)")
plt.imshow(OD_crop_thresh)
plt.colorbar()

plt.tight_layout()
plt.show()